In [6]:
# Cell 1 — Install
!pip install groq pandas -q
print('✅ Install selesai')

✅ Install selesai


In [9]:
# Cell 2 — Imports & Konfigurasi
import os, re, time
import pandas as pd
from groq import Groq

# ── Sesuaikan path & API key ────────────────────────────
INPUT_CSV    = '/content/gopay (2).csv'   # upload CSV lama ke Colab dulu
OUTPUT_CSV   = 'gopay_relabeled.csv'          # output baru dengan label bersih
GROQ_API_KEY = 'your_groq_api_key_here'      # ganti dengan API key Groq kamu
MODEL        = 'llama-3.1-8b-instant'
DELAY        = 0.5                            # detik antar request
# ────────────────────────────────────────────────────────

client = Groq(api_key=GROQ_API_KEY)
print('✅ Konfigurasi siap')


✅ Konfigurasi siap


In [10]:
# Cell 3 — Fungsi Text Cleansing
def clean_tweet(text: str) -> str:
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)   # hapus URL
    text = re.sub(r'@\w+', '', text)                  # hapus mention
    text = re.sub(r'#(\w+)', r'\1', text)            # hapus simbol #, simpan kata
    text = re.sub(r'\bRT\b', '', text)               # hapus RT
    text = re.sub(r'[^\w\s]', ' ', text)             # hapus tanda baca & simbol
    text = re.sub(r'\d+', '', text)                   # hapus angka
    text = re.sub(r'\s+', ' ', text).strip()          # normalisasi spasi
    text = text.lower()
    return text


# Preview cleansing pada 5 sampel
df_preview = pd.read_csv(INPUT_CSV)
print('Contoh cleansing:\n')
for raw in df_preview['full_text'].head(5):
    cleaned = clean_tweet(raw)
    print(f'  BEFORE: {str(raw)[:90]}')
    print(f'  AFTER : {cleaned[:90]}')
    print()

Contoh cleansing:

  BEFORE: @tanyarlfes lah itu udah dksh ovo ma gopay... uang digital ini. brp sih dptnya? klo masing
  AFTER : lah itu udah dksh ovo ma gopay uang digital ini brp sih dptnya klo masing cuman k ya wajar

  BEFORE: @tanyarlfes Buat yang sudah marah2x mohon di lingat ya sender diatas kurang detail. 1. Uan
  AFTER : buat yang sudah marahx mohon di lingat ya sender diatas kurang detail uang operasional sep

  BEFORE: @hehe_ajadulu @poerbaa_ @tanyarlfes Ovo ama Gopay uang digital? Uang juga gak sih?
  AFTER : ovo ama gopay uang digital uang juga gak sih

  BEFORE: @selowbeud @ohcumanspeak @MaulanaTeten @tanyarlfes Kan Ovo Gopay Baju Snack kurang apa lag
  AFTER : kan ovo gopay baju snack kurang apa lagi oh iya ovo dan gopay kan uang digital bukan arti 

  BEFORE: @paw_jhope @sosmedkeras Bukan COD solusinya. Kalau COD driver hrs bolak balik untuk antar 
  AFTER : bukan cod solusinya kalau cod driver hrs bolak balik untuk antar uangnya hrs nya wajib pak



In [11]:
# Cell 4 — Fungsi Labeling via Groq
SYSTEM_PROMPT = (
    'Kamu adalah sistem anotasi sentimen untuk teks Bahasa Indonesia.\n'
    'Tugasmu adalah mengklasifikasikan sentimen tweet terhadap layanan GoPay.\n\n'
    'Aturan klasifikasi:\n'
    '- "positif" : tweet mengekspresikan kepuasan, pujian, atau pengalaman baik terhadap GoPay\n'
    '- "negatif" : tweet mengekspresikan keluhan, kekecewaan, atau pengalaman buruk terhadap GoPay\n'
    '- "netral"  : tweet informatif, pertanyaan, atau tidak jelas sentimennya terhadap GoPay\n\n'
    'PENTING: Balas HANYA dengan satu kata: positif, negatif, atau netral.\n'
    'Jangan tambahkan penjelasan, tanda baca, atau kata lain.'
)


def label_tweet(text: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': f'Tweet: {text}'}
                ],
                max_tokens=10,
                temperature=0.0,
            )
            label = response.choices[0].message.content.strip().lower()
            if label in ['positif', 'negatif', 'netral']:
                return label
            for valid in ['positif', 'negatif', 'netral']:
                if valid in label:
                    return valid
            return 'netral'  # fallback
        except Exception as e:
            print(f'  ⚠️  Error attempt {attempt+1}: {e}')
            time.sleep(2 ** attempt)
    return 'netral'


print('✅ Fungsi label_tweet siap')

✅ Fungsi label_tweet siap


In [12]:
# Cell 5 — Labeling ulang dari nol dengan teks bersih
df = pd.read_csv(INPUT_CSV)
df['clean_text'] = df['full_text'].apply(clean_tweet)

# Hapus baris yang jadi kosong setelah cleansing
before = len(df)
df = df[df['clean_text'].str.strip().str.len() > 3].copy()
print(f'Baris valid setelah cleansing : {len(df)} (dihapus: {before - len(df)})')

# Reset semua label ke None
df['label'] = None

total = len(df)
print(f'\nMulai labeling {total} tweets dengan teks bersih...')
print('=' * 60)

for i, (idx, row) in enumerate(df.iterrows()):
    text  = row['clean_text'][:500]
    label = label_tweet(text)
    df.at[idx, 'label'] = label

    print(f'[{i+1:3d}/{total}] {label:8s} | {text[:65]}...')

    # Auto-save setiap 10 tweet
    if (i + 1) % 10 == 0:
        df.to_csv(OUTPUT_CSV, index=False)
        print(f'  💾 Auto-saved ({i+1}/{total})\n')

    time.sleep(DELAY)

# Simpan final
df.to_csv(OUTPUT_CSV, index=False)

print('\n' + '=' * 60)
print('  ✅ LABELING SELESAI')
print('=' * 60)
print(f'Output : {OUTPUT_CSV}')
print(f'\nDistribusi label:')
print(df['label'].value_counts().to_string())
print(f'\nTotal  : {len(df)} tweets')
print('=' * 60)

Baris valid setelah cleansing : 481 (dihapus: 0)

Mulai labeling 481 tweets dengan teks bersih...
[  1/481] netral   | lah itu udah dksh ovo ma gopay uang digital ini brp sih dptnya kl...
[  2/481] netral   | buat yang sudah marahx mohon di lingat ya sender diatas kurang de...
[  3/481] netral   | ovo ama gopay uang digital uang juga gak sih...
[  4/481] netral   | kan ovo gopay baju snack kurang apa lagi oh iya ovo dan gopay kan...
[  5/481] netral   | bukan cod solusinya kalau cod driver hrs bolak balik untuk antar ...
[  6/481] positif  | streaming klo mau menang titik punya uang receh uang lebih di atm...
[  7/481] netral   | ditunggu oleh oleh nya bang dijadikan dalam bentuk uang digital j...
[  8/481] netral   | jika tidak butuh uang cash tapi butuh pembayaran menggunakan domp...
[  9/481] negatif  | kenapa sih alfamart kebanyakan malas kali transaksi uang digital ...
[ 10/481] netral   | rp kalo ad yg berbaik hati saya nerima uang digital berupa gopay ...
  💾 Auto-saved (10/481)